In [ ]:
# CELL 1 — Imports, Reproducibility, Setup
import sys
import os
import numpy as np
import random
import tensorflow as tf

# Force CPU only
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Reproducibility settings applied (CPU mode).")

# General imports
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
from math import atan2, asin
from scipy.signal import butter, filtfilt
import time

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, precision_recall_curve

from tensorflow import keras
from tensorflow.keras import layers

# Dataset paths
RAW_ROOT = Path("wide_data")
CLEAN_ROOT = Path("cleaned_data_lstm")
CLEAN_ROOT.mkdir(exist_ok=True)

# Quick sanity checks
print("Local dataset paths set.")
print("RAW_ROOT  =", RAW_ROOT.resolve())
print("CLEAN_ROOT=", CLEAN_ROOT.resolve())

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"wide_data folder not found at: {RAW_ROOT.resolve()}\n"
        "Put wide_data in the same directory as the notebook, or adjust RAW_ROOT."
    )

print("Dataset ready (local).")

In [ ]:
# CELL 2 — Sensor Feature Definitions + Quaternion Math
SENSORS = ["L5", "T4", "C7", "T12"]

ACC_FEATURES  = ["Acceleration X(g)", "Acceleration Y(g)", "Acceleration Z(g)"]
GYRO_FEATURES = ["Angular velocity X(°/s)", "Angular velocity Y(°/s)", "Angular velocity Z(°/s)"]
QUAT_FEATURES = ["Quaternions 0()", "Quaternions 1()", "Quaternions 2()", "Quaternions 3()"]

MAG_WEAK_PREFIX = "Magnetic field"

POSTURES = sorted([p.name for p in RAW_ROOT.iterdir() if p.is_dir()])
LABEL_TO_ID = {p: i for i, p in enumerate(POSTURES)}

print("Detected Postures:", POSTURES)
print("Label Map:", LABEL_TO_ID)


# -------------------------
# Magnetometer detection
# -------------------------
def detect_mag_cols(df, sensor):
    cols = df.columns
    out = []

    for axis in ["x", "y", "z"]:
        matches = [
            c for c in cols
            if c.lower().startswith(sensor.lower() + "_magnetic field")
            and axis in c.lower()
        ]
        if not matches:
            raise KeyError(f" Missing magnetometer axis={axis} for sensor={sensor}\n{list(cols)}")

        out.append(matches[0])

    return out


# -------------------------
# Quaternion helpers
# -------------------------
def quat_reorder_to_wxyz(qx, qy, qz, qw):
    return np.array([qw, qx, qy, qz], dtype=np.float32)

def quat_conjugate(q):
    w,x,y,z = q
    return np.array([w, -x, -y, -z], dtype=np.float32)

def quat_multiply(q1, q2):
    w1,x1,y1,z1 = q1
    w2,x2,y2,z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dtype=np.float32)

def rotate_vector_by_quaternion(q, v):
    vq = np.array([0, v[0], v[1], v[2]], dtype=np.float32)
    return quat_multiply(quat_multiply(q, vq), quat_conjugate(q))[1:]

def quaternion_to_euler(q):
    w,x,y,z = q
    yaw   = atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    pitch = asin(np.clip(2*(w*y - z*x), -1, 1))
    roll  = atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    return yaw, pitch, roll

In [ ]:
# CELL 3 — Preprocessing Filters
def hampel_filter(x, window_size=5, n_sigmas=3):
    x = x.copy()
    y = x.copy()
    N = len(x)
    k = window_size

    for i in range(k, N-k):
        window = x[i-k:i+k+1]
        median = np.nanmedian(window)
        mad = np.nanmedian(np.abs(window - median))

        if mad < 1e-6:
            continue

        if abs(x[i] - median) > n_sigmas * 1.4826 * mad:
            y[i] = median

    return y

def butter_lowpass_filter(x, cutoff=3.0, fs=50, order=2):
    b,a = butter(order, cutoff/(0.5*fs), btype="low")
    try:
        return filtfilt(b,a,x,axis=0)
    except:
        return x

def remove_bias(x):
    return x - np.nanmean(x, axis=0)

In [ ]:
# CELL 4 — Full Preprocessing Pipeline
def preprocess_single_file(raw_path):
    """
    Preprocess a single raw CSV file and return the processed array (ALL FEATURES).
    Does NOT write to disk.
    """
    df = pd.read_csv(raw_path)

    # Keep relevant columns only
    keep = []
    for col in df.columns:
        for s in SENSORS:
            if col.startswith(f"{s}_Acceleration"):     keep.append(col)
            elif col.startswith(f"{s}_Angular velocity"): keep.append(col)
            elif col.startswith(f"{s}_Quaternions"):     keep.append(col)
            elif col.startswith(f"{s}_Magnetic field"):  keep.append(col)

    df = df[keep].copy().apply(pd.to_numeric, errors="coerce")

    df.replace([np.inf,-np.inf], np.nan, inplace=True)
    df.interpolate(limit_direction="both", inplace=True)
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    blocks = []

    for sensor in SENSORS:

        acc = df[[f"{sensor}_{f}" for f in ACC_FEATURES]].values
        gyr = df[[f"{sensor}_{f}" for f in GYRO_FEATURES]].values
        quat_raw = df[[f"{sensor}_{f}" for f in QUAT_FEATURES]].values
        mag_cols = detect_mag_cols(df, sensor)
        mag = df[mag_cols].values

        # Quaternion normalization
        quats = []
        for (qx,qy,qz,qw) in quat_raw:
            q = quat_reorder_to_wxyz(qx,qy,qz,qw)
            q = q / np.linalg.norm(q)
            quats.append(q)
        quats = np.array(quats)

        # Remove bias
        acc = remove_bias(acc)
        gyr = remove_bias(gyr)
        mag = remove_bias(mag)

        # Hampel filtering
        for i in range(acc.shape[1]): acc[:,i] = hampel_filter(acc[:,i])
        for i in range(gyr.shape[1]): gyr[:,i] = hampel_filter(gyr[:,i])
        for i in range(mag.shape[1]): mag[:,i] = hampel_filter(mag[:,i])

        # Low-pass filter
        acc = butter_lowpass_filter(acc)
        gyr = butter_lowpass_filter(gyr)
        mag = butter_lowpass_filter(mag)

        # Orientation alignment + Euler
        A,G,M,E = [],[],[],[]

        for t in range(len(df)):
            q = quats[t]
            A.append(rotate_vector_by_quaternion(q, acc[t]))
            G.append(rotate_vector_by_quaternion(q, gyr[t]))
            M.append(rotate_vector_by_quaternion(q, mag[t]))
            E.append(quaternion_to_euler(q))

        A = np.array(A); G = np.array(G); M = np.array(M); E = np.array(E)

        A_mag = np.linalg.norm(A,axis=1,keepdims=True)
        G_mag = np.linalg.norm(G,axis=1,keepdims=True)
        M_mag = np.linalg.norm(M,axis=1,keepdims=True)

        block = np.concatenate([A,G,M,A_mag,G_mag,M_mag,E], axis=1)
        blocks.append(block)

    X_out = np.concatenate(blocks, axis=1)

    return X_out

In [ ]:
# CELL 5 — Load and Preprocess All Data
print("\n===== PREPROCESSING ALL FILES =====\n")

SUBJECT_RE = re.compile(r"Sub_(\d+)_Trial_(\d+)\.csv")

rows = []

for posture_dir in RAW_ROOT.iterdir():
    if not posture_dir.is_dir():
        continue

    files = list(posture_dir.glob("*.csv"))
    print(f"Processing posture: {posture_dir.name} ({len(files)} files)")

    for raw_file in files:
        # Preprocess in-memory (no disk I/O for cleaned files)
        X_processed = preprocess_single_file(raw_file)
        
        # Extract subject and trial info
        m = SUBJECT_RE.search(raw_file.name)
        if m:
            rows.append({
                "data": X_processed,
                "posture": posture_dir.name,
                "label": LABEL_TO_ID[posture_dir.name],
                "subject": int(m.group(1)),
                "trial": int(m.group(2)),
                "filename": raw_file.name
            })

df_items = pd.DataFrame(rows)

print(f"\nTotal processed files: {len(df_items)}")
print(f"Unique subjects: {df_items['subject'].nunique()}")
print(f"Feature dimension: {df_items['data'].iloc[0].shape[1]}")

In [ ]:
# CELL 6 — Windowing
FS = 50
WIN_LEN = 200
STRIDE = 100

print(f"\nWindow length = {WIN_LEN} frames ({WIN_LEN/FS:.2f} sec)")
print(f"Stride = {STRIDE} frames")

def make_windows(X):
    """
    Convert a full sequence (T,F) into windows (num_windows, WIN_LEN, F)
    with overlap.
    """
    T, F = X.shape

    # If trial is shorter than window, pad with zeros
    if T < WIN_LEN:
        w = np.zeros((WIN_LEN, F))
        w[:T] = X
        return w[None,:,:]

    windows = []
    for i in range(0, T - WIN_LEN + 1, STRIDE):
        windows.append(X[i:i+WIN_LEN])

    return np.array(windows)

In [ ]:
# CELL 7 — Create Window Dataset (ACC + EULER ONLY)
X_all = []
y_all = []
subj_all = []
trial_all = []

window_counter = 0

for _, row in df_items.iterrows():

    X_full = row["data"]  # Use preprocessed data from memory

    # ---------------------------------
    # ACC + EULER (6 features per sensor)
    # Extract from the 60-feature output:
    # L5: features 0-14 → Acc (0-2) + Euler (12-14)
    # T4: features 15-29 → Acc (15-17) + Euler (27-29)
    # C7: features 30-44 → Acc (30-32) + Euler (42-44)
    # T12: features 45-59 → Acc (45-47) + Euler (57-59)
    # ---------------------------------
    acc_euler_slices = [
        np.r_[0:3, 12:15],     # L5
        np.r_[15:18, 27:30],   # T4
        np.r_[30:33, 42:45],   # C7
        np.r_[45:48, 57:60]    # T12
    ]

    X = np.concatenate([X_full[:, s] for s in acc_euler_slices], axis=1)

    windows = make_windows(X)

    for w in windows:
        X_all.append(w)
        y_all.append(row["label"])
        subj_all.append(row["subject"])
        trial_all.append(row["trial"])

    window_counter += len(windows)

X_all = np.array(X_all)
y_all = np.array(y_all)
subj_all = np.array(subj_all)
trial_all = np.array(trial_all)

print("\n===== DATASET SUMMARY =====")
print("Total windows:", window_counter)
print("X shape:", X_all.shape)  # should be (N, 200, 24)
print("Features per timestep:", X_all.shape[2])

In [ ]:
# CELL 8 — Normalization (ACC + EULER → 6 per sensor)
SENSOR_SLICES = [
    slice(0,6),
    slice(6,12),
    slice(12,18),
    slice(18,24)
]

def compute_sensor_norm(X):

    means = []
    stds  = []

    for s in SENSOR_SLICES:

        sensor_data = X[:,:,s]
        flat = sensor_data.reshape(-1, sensor_data.shape[-1])

        mean = flat.mean(axis=0, keepdims=True)
        std  = flat.std(axis=0, keepdims=True) + 1e-8

        means.append(mean)
        stds.append(std)

    return means, stds


def apply_sensor_norm(X, means, stds):

    Xn = X.copy()

    for s, mean, std in zip(SENSOR_SLICES, means, stds):
        Xn[:,:,s] = (X[:,:,s] - mean) / std

    return Xn

In [ ]:
# CELL 9 — SensorDropout Layer
class SensorDropout(layers.Layer):

    def __init__(self, drop_prob=0.2):
        super().__init__()
        self.drop_prob = drop_prob

    def call(self, x, training=None):

        if not training:
            return x

        batch = tf.shape(x)[0]
        n_sensors = 4
        feat_per_sensor = tf.shape(x)[-1] // n_sensors

        mask = tf.cast(
            tf.random.uniform((batch, n_sensors, 1)) > self.drop_prob,
            x.dtype
        )

        mask = tf.repeat(mask, repeats=feat_per_sensor, axis=2)
        mask = tf.reshape(mask, (batch, 1, n_sensors * feat_per_sensor))

        return x * mask

In [ ]:
# CELL 10 — Model with Attention Extraction
def build_model(input_shape):

    inp = keras.Input(shape=input_shape)

    print("Model input:", input_shape)

    x = SensorDropout(0.25)(inp)

    # ---------------------------------
    # Split sensors (ACC + EULER → 6 each)
    # ---------------------------------

    L5  = layers.Lambda(lambda x: x[:,:,0:6])(x)
    T4  = layers.Lambda(lambda x: x[:,:,6:12])(x)
    C7  = layers.Lambda(lambda x: x[:,:,12:18])(x)
    T12 = layers.Lambda(lambda x: x[:,:,18:24])(x)

    print("Sensor feature size:", 6)

    # ---------------------------------
    # Sensor CNN block WITH NAMED ATTENTION
    # ---------------------------------

    def sensor_block(x, name_prefix):

        x = layers.Conv1D(32, 5, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)

        attention = layers.Dense(32, activation="tanh", name=f"{name_prefix}_feat_dense")(x)
        attention = layers.Softmax(axis=-1, name=f"{name_prefix}_feat_softmax")(attention)

        x = layers.Multiply(name=f"{name_prefix}_feat_mul")([x, attention])

        return x, attention

    L5,  L5_feat_att  = sensor_block(L5,  "L5")
    T4,  T4_feat_att  = sensor_block(T4,  "T4")
    C7,  C7_feat_att  = sensor_block(C7,  "C7")
    T12, T12_feat_att = sensor_block(T12, "T12")

    # ---------------------------------
    # Stack sensors
    # ---------------------------------

    sensors = layers.Lambda(lambda x: tf.stack(x, axis=2))([L5, T4, C7, T12])

    # ---------------------------------
    # Sensor attention 
    # ---------------------------------

    sensor_attention = layers.Dense(1, activation="tanh", name="sensor_dense")(sensors)
    sensor_attention = layers.Softmax(axis=2, name="sensor_softmax")(sensor_attention)

    sensors = layers.Multiply()([sensors, sensor_attention])

    fused = layers.Lambda(lambda x: tf.reduce_sum(x, axis=2))(sensors)

    # ---------------------------------
    # Temporal pooling
    # ---------------------------------

    x = layers.GlobalAveragePooling1D()(fused)

    # ---------------------------------
    # Classifier
    # ---------------------------------

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(len(POSTURES), activation="softmax")(x)

    model = keras.Model(inp, out)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    # IMPORTANT: attach attention tensors
    model.feature_attentions = [L5_feat_att, T4_feat_att, C7_feat_att, T12_feat_att]
    model.sensor_attention = sensor_attention

    model.summary()

    return model

In [ ]:
# CELL 11 — Learning Rate Scheduler
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# CELL 12 — LOSO Loop (Per-Fold Only)
from collections import defaultdict
import gc

EXCLUDED_SUBJECTS = [16, 21, 26, 29]

subjects = sorted(
    s for s in np.unique(subj_all)
    if s not in EXCLUDED_SUBJECTS
)

window_scores = []
trial_scores = []

all_window_true = []
all_window_pred = []

all_trial_true = []
all_trial_pred = []
all_trial_subjects = []

# ✅ store probs + attention inputs
all_window_probs = []
all_attention_inputs = []   # 🔥 NEW

fold_results = []

for test_subject in subjects:

    print("\n============================")
    print("TEST SUBJECT:", test_subject)
    print("============================")

    # -----------------------
    # SPLIT
    # -----------------------
    train_subjects = [s for s in subjects if s != test_subject]

    val_size = max(1, int(0.2 * len(train_subjects)))
    val_subjects = np.random.choice(train_subjects, size=val_size, replace=False)

    train_subjects_final = [s for s in train_subjects if s not in val_subjects]

    train_mask = np.isin(subj_all, train_subjects_final)
    val_mask   = np.isin(subj_all, val_subjects)
    test_mask  = subj_all == test_subject

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]

    X_val   = X_all[val_mask]
    y_val   = y_all[val_mask]

    X_test  = X_all[test_mask]
    y_test  = y_all[test_mask]

    trial_test = trial_all[test_mask]
    subject_test = subj_all[test_mask]

    # -----------------------
    # NORMALIZATION
    # -----------------------
    means, stds = compute_sensor_norm(X_train)

    X_train = apply_sensor_norm(X_train, means, stds)
    X_val   = apply_sensor_norm(X_val, means, stds)
    X_test  = apply_sensor_norm(X_test, means, stds)

    # store normalized test for attention
    all_attention_inputs.append(X_test)

    # -----------------------
    # ONE HOT
    # -----------------------
    y_train_cat = keras.utils.to_categorical(y_train, len(POSTURES))
    y_val_cat   = keras.utils.to_categorical(y_val, len(POSTURES))

    # -----------------------
    # MODEL
    # -----------------------
    model = build_model(X_train.shape[1:])

    early = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    hist = model.fit(
        X_train,
        y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=60,
        batch_size=32,
        callbacks=[early, lr_scheduler],
        verbose=1
    )

    # -----------------------
    # WINDOW LEVEL
    # -----------------------
    probs = model.predict(X_test, verbose=0)

    y_pred = np.argmax(probs, axis=1)
    y_true = y_test

    all_window_probs.append(probs)

    window_acc = np.mean(y_pred == y_true)
    window_scores.append(window_acc)

    print("\nWINDOW ACC:", window_acc)
    print(classification_report(y_true, y_pred, target_names=POSTURES))

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
    plt.title(f"Window CM - Subject {test_subject}")
    plt.show()

    all_window_true.extend(y_true)
    all_window_pred.extend(y_pred)

    # -----------------------
    # TRIAL LEVEL
    # -----------------------
    trial_probs = defaultdict(list)
    trial_true = {}

    for p,true,s,t in zip(probs, y_true, subject_test, trial_test):
        key = (s,t)
        trial_probs[key].append(p)
        trial_true[key] = true

    trial_pred = []
    trial_true_list = []
    trial_subjects = []

    for k in trial_probs:
        w = np.array(trial_probs[k])
        summed = np.sum(np.log(w + 1e-8), axis=0)
        pred = np.argmax(summed)

        trial_pred.append(pred)
        trial_true_list.append(trial_true[k])
        trial_subjects.append(k[0])

    trial_pred = np.array(trial_pred)
    trial_true_list = np.array(trial_true_list)
    trial_subjects = np.array(trial_subjects)

    trial_acc = np.mean(trial_pred == trial_true_list)
    trial_scores.append(trial_acc)

    print("\nTRIAL ACC:", trial_acc)
    print(classification_report(trial_true_list, trial_pred, target_names=POSTURES))

    cm = confusion_matrix(trial_true_list, trial_pred)
    ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
    plt.title(f"Trial CM - Subject {test_subject}")
    plt.show()

    all_trial_true.extend(trial_true_list)
    all_trial_pred.extend(trial_pred)
    all_trial_subjects.extend(trial_subjects)

    # SAVE FINAL MODEL
    if test_subject == subjects[-1]:
        final_model = model
        final_means = means
        final_stds = stds

    del model, hist
    gc.collect()

In [ ]:
# CELL 13 — Final Results (All Folds)
print("\n===== FINAL LOSO RESULTS =====")

print("Avg Window Acc:", np.mean(window_scores))
print("Avg Trial Acc:", np.mean(trial_scores))

# =========================================
# FIX PROB SHAPE
# =========================================
all_window_probs = np.concatenate(all_window_probs, axis=0)

print("Probs shape:", all_window_probs.shape)
print("True shape:", np.array(all_window_true).shape)

# -----------------------
# WINDOW METRICS
# -----------------------
print("\nWINDOW METRICS")
print(classification_report(all_window_true, all_window_pred, target_names=POSTURES))

cm = confusion_matrix(all_window_true, all_window_pred)
ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Window CM (LOSO)")
plt.show()

# -----------------------
# TRIAL METRICS
# -----------------------
print("\nTRIAL METRICS")
print(classification_report(all_trial_true, all_trial_pred, target_names=POSTURES))

cm = confusion_matrix(all_trial_true, all_trial_pred)
ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Trial CM (LOSO)")
plt.show()

# =========================================
# SUBJECT LEVEL (VOTING)
# =========================================

subject_votes = defaultdict(list)
subject_true = {}

for pred, true, s in zip(all_trial_pred, all_trial_true, all_trial_subjects):
    key = (s, true)
    subject_votes[key].append(pred)
    subject_true[key] = true

all_subject_pred = []
all_subject_true = []

for k in subject_votes:
    votes = np.array(subject_votes[k])
    pred = np.bincount(votes).argmax()

    all_subject_pred.append(pred)
    all_subject_true.append(subject_true[k])

all_subject_pred = np.array(all_subject_pred)
all_subject_true = np.array(all_subject_true)

print("\n===== SUBJECT RESULTS =====")
print("Avg Subject Acc:", np.mean(all_subject_pred == all_subject_true))

# -----------------------
# SUBJECT METRICS
# -----------------------
print("\nSUBJECT METRICS")
print(classification_report(
    all_subject_true,
    all_subject_pred,
    labels=np.arange(len(POSTURES)),
    target_names=POSTURES,
    zero_division=0
))

# -----------------------
# SUBJECT CONFUSION MATRIX
# -----------------------
cm = confusion_matrix(
    all_subject_true,
    all_subject_pred,
    labels=np.arange(len(POSTURES))
)

ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Subject Confusion Matrix (LOSO)")
plt.show()

In [ ]:
# CELL 14 — Real Deployment Latency
sample = X_all[0:1]
sample = apply_sensor_norm(sample, final_means, final_stds)

for _ in range(10):
    final_model.predict(sample, verbose=0)

latencies = []
N = 100

for _ in range(N):
    start = time.perf_counter()
    final_model.predict(sample, verbose=0)
    end = time.perf_counter()
    latencies.append(end - start)

latencies = np.array(latencies)

print("\n===== LATENCY =====")
print(f"Mean: {latencies.mean()*1000:.3f} ms")
print(f"Std: {latencies.std()*1000:.3f} ms")

In [ ]:
# CELL 15 — ROC Curve
y_true_bin = label_binarize(all_window_true, classes=np.arange(len(POSTURES)))

plt.figure(figsize=(8,6))

for i in range(len(POSTURES)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_window_probs[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{POSTURES[i]} (AUC={roc_auc:.3f})")

plt.plot([0,1],[0,1],'k--')
plt.title("ROC Curve (LOSO)")
plt.legend()
plt.grid()
plt.show()

print("\n===== ROC AUC SCORES =====")
for i in range(len(POSTURES)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_window_probs[:, i])
    print(f"{POSTURES[i]}: {auc(fpr, tpr):.4f}")

In [ ]:
# CELL 16 — PR Curve
plt.figure(figsize=(8,6))

for i in range(len(POSTURES)):
    precision, recall, _ = precision_recall_curve(
        y_true_bin[:, i],
        all_window_probs[:, i]
    )

    pr_auc = auc(recall, precision)

    plt.plot(recall, precision, label=f"{POSTURES[i]} (AUC={pr_auc:.3f})")

plt.title("Precision-Recall Curve (LOSO)")
plt.legend()
plt.grid()
plt.show()